# 09 · Explore CellBridge predictions in tissue

Review saved **CellBridge** predictions against cell morphology. Start with **NuCLS** or **PhenoCell**; the HEST tissue map and TERRA views are optional. All views use **branch graph** conformal predictions.

Use the lab's **hest** kernel. Run the setup and explore cells in order, then choose a dataset, field and patch. No GPU is needed.

## Reading the plots

| Term | Meaning |
|---|---|
| Reference label | The harmonized dataset annotation; it may contain errors. |
| Predicted label | CellBridge's highest-scoring class. |
| Prediction set | Candidate labels returned by the conformal method. |
| Common ancestor (LCA) | The most specific shared parent of the candidates in the Cell Ontology. |
| Ancestor-expanded set | Evaluated classes beneath the chosen ancestor; may add candidates. |
| Depth / coarseness | Shortest steps **up from the reference term to the ancestor**. **0 = same term; larger = broader.** Requires the reference label; it is not a confidence probability. |

**Black ×:** ancestor excludes the reference. **Red +:** empty prediction set. Both have no depth. **Hollow grey circle:** no saved test prediction. Markers show annotated cell centres.

Coverage is the fraction containing the reference label. The nominal target applies at population level; selected patches can differ.

In [ ]:
import importlib
import sys
from pathlib import Path

# Find the checkout when launched from the repository, notebooks/, or workspace.
REPO = next((p for base in (Path.cwd(), *Path.cwd().parents)
             for p in (base, base / "projects/cellbridge")
             if (p / "src/cellbridge/config.py").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Launch Jupyter from the cellbridge checkout.")
sys.path.insert(0, str(REPO / "src"))
from cellbridge.qualitative import data, figures, views

# Rerunning setup picks up edits even when the kernel is already running.
for module in (data, figures, views):
    importlib.reload(module)

## Available datasets

Menus show saved predictions only. Fields are ordered by evaluated-cell count.

In [ ]:
catalog = views.show_catalog()

## Saving your review

**Save displayed view** exports the rendered figure, cell or patch tables, and provenance to:

```text
<workspace>/outputs/reproducibility/figures/qualitative_conformal/<username>/<dataset>/<timestamp>/
```

Each save creates a new folder. After changing controls, render again before saving.

## Explore

1. **Load dataset**, then choose a **Field** and **Patch**. PhenoCell's first load can take several minutes.
2. Choose **Patch** or **Cell**. **Find cell** searches IDs and original annotations; enable **Cells from whole field** to search beyond the patch.
3. Click **Show selection**, inspect the panels, then **Save displayed view**.

Optional **Smooth depth map** shows a local mean over covered cells only. Excluded and empty sets keep their markers. Depth colours use a fixed scale within each dataset/section.

In [ ]:
explorer = views.Explorer(catalog).show()

## HEST tissue map

1. Choose a **Section**, **Statistic**, **Patch (px)** and **Min cells**, then click **Show tissue map**.
2. Hover a painted patch for its ID and value. The **Patch** menu lists patches worst first; choose one and click **Open selected patch**.
3. **Save displayed view** saves the map and patch table, plus the opened patch when present.

**Red marks the worse end.** Mean/median depth uses covered cells only; check excluded and empty-set fractions alongside it. Coverage and accuracy use all evaluated cells in the patch.

Only patches meeting **Min cells** are painted, within the held-out test region. Raise that threshold for more stable summaries. Depth uses the section's fixed ontology scale; other statistics rescale within each map, so compare their numeric values across maps. Patch sizes are in original image pixels; figures also show µm.

In [ ]:
tissue_map = views.TissueMap(catalog).show()

## TERRA transcriptomic view

Compare the selected HEST patch with embeddings of its Xenium transcript counts. CellBridge uses H&E; TERRA provides a transcriptomic comparison.

1. Choose a patch in the tissue map above.
2. Select **Embedding**, **Colour by** and **Context**, then click **Fetch TERRA embeddings**. Larger contexts can take several minutes.
3. **Save displayed view** exports the figure, patch cells, UMAP coordinates and settings.

| Panel | Interpretation |
|---|---|
| UMAP | Patch cells in colour; sampled section context in grey. Compare local neighbourhoods, not distances between separated groups. |
| Tissue | The same patch cells and colours on H&E. |
| Neighbourhood agreement | Mean fraction of up to 15 nearest evaluated neighbours sharing the reference label, grouped by depth. Neighbours are measured in TERRA PCA space. |

HEST reference labels also come from transcript counts, so this is **supporting context, not independent validation**. Agreement depends on the sampled context. UMAP is refitted on each click; layouts across different settings are not directly comparable.

In [ ]:
terra = views.TerraView(tissue_map).show()

## Review notes

For each saved example, record its folder and a short observation:

- Does the predicted label fit the morphology and neighbours?
- Does the candidate set contain the reference? Is the ancestor biologically useful?
- Why did you select this example? Include exclusions and failures as well as successes.

The displayed crop may differ from the model's input crop. Selected examples do not estimate whole-dataset performance.